# Data Quality Audit — 06 · Booking.com Hotels (Kaggle)

**Source:** `data/raw/Booking.com/project3_df1.csv`

**Purpose in the concierge:** Accommodation: price, ratings, rooms, coordinates.

Standardized audit covering:

```
Dataset
├── Shape
├── Columns & data types
├── Missing values
├── Duplicates
├── Invalid values
├── Outliers
├── Inconsistent categories
├── Geographic validity
├── Date/time validity
├── Data-source/license
└── Known limitations
```

> **Snapshot:** booking links carry `checkin=2020-04-24` → **2020 data**; prices are historical.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

# Resolve repo root whether run from the audit folder or the repo root.
p = Path.cwd()
while p != p.parent and not (p / "data" / "raw").exists():
    p = p.parent
ROOT = p
print("repo root:", ROOT)

# Saudi Arabia bounding box (approx) for geographic validity checks.
SA_LAT = (16.0, 32.5)
SA_LON = (34.5, 56.0)

def iqr_outliers(series):
    """Return (count, lower, upper) of IQR outliers in a numeric series."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return 0, np.nan, np.nan
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return int(((s < lo) | (s > hi)).sum()), round(lo, 2), round(hi, 2)

def missing_report(df):
    m = pd.DataFrame({"missing": df.isna().sum(),
                      "missing_%": (df.isna().mean() * 100).round(1)})
    return m[m["missing"] > 0].sort_values("missing", ascending=False)

def dtype_report(df):
    return pd.DataFrame({
        "dtype": [str(t) for t in df.dtypes],
        "non_null": df.notna().sum().values,
        "n_unique": [df[c].nunique(dropna=True) for c in df.columns],
    }, index=df.columns)


repo root: /home/user/saudi-Digital-Concierge


In [2]:
df = pd.read_csv(ROOT / "data/raw/Booking.com/project3_df1.csv").rename(columns={"Unnamed: 0":"idx"})
df["price_sar"] = pd.to_numeric(df["Price"].str.replace("SAR","",regex=False).str.replace(",","").str.strip(), errors="coerce")
print("Loaded hotels:", df.shape)
df.head(3)

Loaded hotels: (1025, 22)


,idx,Name,City,Price,Star_Rating,Property_Demand,Property_id,Customers_Rating,Customers_Review,Type_of_room,reservations_Payment,Canelation,Max_persons,Bed_type,Tax,Review_title,Credit_card,Breakfst_included,Longitude_x,Latitude_y,Link,price_sar
0,0,الريـم,Al Ula,SAR 179,5,Only 1 room left like this on our site,6330099,NaN,NaN,Economy Double Room,No prepayment needed,FREE cancellation,Max persons: 2,3 beds\n(3 large doubles),+SAR 0 taxes and charges,NaN,Reservation possible without a credit card,NaN,37.917525,26.648656,https://www.booking.com/hotel/sa/lrym.en-gb.ht...,179
1,1,Copper Crown Furnished Apartments,Khamis Mushayt,SAR 195,5,Only 2 rooms like this left on our site,5326174,9.0,169 reviews,Deluxe Room (2 Adults + 1 Child),No prepayment needed,FREE cancellation,Max persons: 2,1 bed\n(1 extra-large double),includes taxes and charges,Superb,Reservation possible without a credit card,NaN,42.801402,18.242741,https://www.booking.com/hotel/sa/kwbr-krwn-lls...,195
2,2,فندق راية الشلال 2,Abū Qa‘ar,SAR 200,5,Only 3 rooms like this left on our site,5987844,8.7,17 reviews,Deluxe Double or Twin Room,No prepayment needed,FREE cancellation,Max persons: 2,1 bed\n(1 double),+SAR 0 taxes and charges,Fabulous,Reservation possible without a credit card,NaN,45.990981,28.370666,https://www.booking.com/hotel/sa/fndq-ry-lshll...,200


## Shape

In [3]:
print("Rows:", len(df), "| Columns:", df.shape[1])

Rows: 1025 | Columns: 22


## Columns & data types
Several columns are mistyped as text (`Price`, `Max_persons`) and some names have typos.

In [4]:
dtype_report(df)

,dtype,non_null,n_unique
idx,int64,1025,1025
Name,str,1025,1024
City,str,1025,112
Price,str,1025,312
Star_Rating,int64,1025,6
Property_Demand,str,750,14
Property_id,int64,1025,1025
Customers_Rating,float64,956,56
Customers_Review,str,956,610
Type_of_room,str,1024,236


## Missing values

In [5]:
missing_report(df.drop(columns='idx'))

,missing,missing_%
Breakfst_included,941,91.8
Credit_card,321,31.3
reservations_Payment,290,28.3
Property_Demand,275,26.8
Canelation,179,17.5
Customers_Rating,69,6.7
Review_title,69,6.7
Customers_Review,69,6.7
Bed_type,19,1.9
Type_of_room,1,0.1


## Duplicates

In [6]:
print("Exact duplicate rows:", df.drop(columns="idx").duplicated().sum())
print("Duplicate Property_id:", df["Property_id"].duplicated().sum())

Exact duplicate rows: 0
Duplicate Property_id: 0


## Invalid values
`Price` should parse to a positive number; `Star_Rating` 0–5 (0 = unrated); `Customers_Rating` 0–10.

In [7]:
print("Price parse fails:", df["price_sar"].isna().sum())
print("Non-positive price:", int((df["price_sar"] <= 0).sum()))
print("Star_Rating out of 0-5:", int((~df["Star_Rating"].between(0,5)).sum()))
print("Star_Rating == 0 (unrated):", int((df["Star_Rating"]==0).sum()))
cr = pd.to_numeric(df["Customers_Rating"], errors="coerce")
print("Customers_Rating out of 0-10:", int(((cr<0)|(cr>10)).sum()))

Price parse fails: 0
Non-positive price: 0
Star_Rating out of 0-5: 0
Star_Rating == 0 (unrated): 532
Customers_Rating out of 0-10: 0


## Outliers
Nightly price is right-skewed.

In [8]:
n, lo, hi = iqr_outliers(df["price_sar"])
print(f"price_sar IQR outliers: {n} (bounds {lo}..{hi})")
print(df["price_sar"].describe().round(0).to_string())

price_sar IQR outliers: 93 (bounds -170.0..550.0)
count    1025.0
mean      269.0
std       451.0
min        23.0
25%       100.0
50%       130.0
75%       280.0
max      7907.0


## Inconsistent categories
`City` mixes neighbourhood + city (`Ajyad, Makkah`); booking-term fields are categorical.

In [9]:
df["base_city"] = df["City"].str.split(",").str[-1].str.strip()
print("Raw City values:", df["City"].nunique(), "| base cities:", df["base_city"].nunique())
print(df["base_city"].value_counts().head(12).to_string())
print("\nBreakfst_included:", df["Breakfst_included"].value_counts(dropna=False).to_dict())

Raw City values: 112 | base cities: 85
base_city
Jeddah        208
Riyadh        169
Makkah         83
Al Khobar      83
Dammam         59
Abha           36
Al Madinah     36
Taif           32
Yanbu          32
Buraydah       31
Jazan          23
Tabuk          23

Breakfst_included: {nan: 941, 'Breakfast included': 84}


## Geographic validity
Coordinates should fall inside the Saudi bounding box (nationwide).

In [10]:
lat = pd.to_numeric(df["Latitude_y"], errors="coerce")
lon = pd.to_numeric(df["Longitude_x"], errors="coerce")
in_sa = lat.between(*SA_LAT) & lon.between(*SA_LON)
print("Missing coords:", lat.isna().sum())
print("Outside Saudi bbox:", int((~in_sa).sum()))
print("Lat range:", round(lat.min(),2), "-", round(lat.max(),2))
print("Lon range:", round(lon.min(),2), "-", round(lon.max(),2))

Missing coords: 0
Outside Saudi bbox: 0
Lat range: 16.85 - 31.68
Lon range: 35.01 - 50.22


## Date/time validity
No per-row date columns; the whole file is a **2020 snapshot** (see license).

In [11]:
print("Date-like columns:", [c for c in df.columns if any(k in c.lower() for k in ["date","time","year"])])

Date-like columns: []


## Data-source / license
- **Source:** Kaggle (Booking.com scrape).
- **License:** TBD.
- **Currency:** **2020 snapshot — not live.** Surface prices as historical.

## Known limitations
- **2020 prices** — historical, not current.
- **`Star_Rating == 0` = unrated** (~52%), not zero-star.
- **Column-name typos** (`Canelation`, `Breakfst_included`, `Longitude_x`, `Latitude_y`).
- `City` mixes neighbourhood + city → extract base city and map to canonical key.
- Sparse columns (`Breakfst_included` ~92% missing).
- Text fields (`Price`, `Max_persons`, `Tax`) need parsing.